## Addendum (v2.3): inference fixes, evidence limits, audit framing, novelty ledger
Run after Section 21. These cells **only read saved results and the dataset file**; they change no model, tuning or split, so the locked LoCoMo test stays locked.
They need `RES_DIR`, and optionally `R`, `raw`, `DATA_PATH` from the notebook; otherwise they load from `memweave2_results/v2_2` and `data/`.

### 23. Corrected inference layer
Holm-correct H2 (duplicate rule removed), test whether ablated components were ever active, drop abstention questions, and test the session-reuse leak path with an exposure gradient.

In [ ]:
# ===== Section 23. Corrected inference layer (reads saved results only; changes no model, no tuning) =====
# Fixes: (a) H2 verdicts must use Holm; (b) duplicate rule counted twice; (c) "SUPPORTED" for components that were never active;
# (d) abstention questions scored as if they had evidence; (e) session-reuse leak path, via an exposure-gradient test.
import json, numpy as np, pandas as pd
from pathlib import Path
from scipy import stats
RES_DIR   = globals().get("RES_DIR", Path("memweave2_results/v2_2"))
DATA_PATH = globals().get("DATA_PATH", Path("data/longmemeval/longmemeval_s_cleaned.json"))
PF_, PRIM = 0.10, "MemWeave-LR (primary)"
R23 = globals().get("R", None)
if R23 is None: R23 = pd.read_csv(RES_DIR / "all_results_raw.csv")
raw23 = globals().get("raw", None)
if raw23 is None: raw23 = json.load(open(DATA_PATH, encoding="utf-8"))
rng23 = np.random.default_rng(23)

def holm23(ps):
    ps = np.asarray(ps, float); o = np.argsort(ps); adj = np.empty_like(ps); run = 0.0
    for r, i in enumerate(o): run = max(run, (len(ps) - r) * ps[i]); adj[i] = min(1.0, run)
    return adj

def wide23(setting="B", frac=PF_, mask=None):
    d = R23[(R23.setting == setting) & np.isclose(R23.budget, frac)]
    w = d.groupby(["qid", "method"]).recall.mean().unstack()
    return w if mask is None else w[mask(w.index)]

def paired23(w, a, b, n_boot=5000):
    d = (w[a] - w[b]).dropna().to_numpy()
    bm = rng23.choice(d, size=(n_boot, len(d))).mean(1)
    try: p = float(stats.wilcoxon(d).pvalue) if np.any(d != 0) else 1.0
    except ValueError: p = 1.0
    return dict(mean_diff=d.mean(), ci_low=np.quantile(bm, .025), ci_high=np.quantile(bm, .975), p=p, n=len(d))

def verdict23(r, alpha=0.05):
    if r.ci_low > 0 and r.holm_p < alpha: return "SUPPORTED (CI excludes 0 and Holm p < .05)"
    if r.ci_low > 0: return "NOT SUPPORTED after Holm (CI excludes 0 but Holm p >= .05): marginal"
    if r.ci_high < 0: return "LOWER (CI excludes 0)"
    return "no reliable difference"

OUT = []
add = OUT.append

# ---- (a)+(b) H2 with Holm, duplicate removed ----
w = wide23()
dup = bool(np.allclose(w["low_novelty"], w["single_feature_cv"]))
FAIR23 = ["low_novelty", "high_centrality", "single_feature_cv_nosim", "heuristic_signed", "rarity_signed",
          "user_density", "heuristic_fill", "heuristic_mmr"]
rows = [dict(comparison=f"{PRIM} - {o}", **paired23(w, PRIM, o)) for o in FAIR23]
H2 = pd.DataFrame(rows); H2["holm_p"] = holm23(H2.p.values); H2["verdict"] = H2.apply(verdict23, axis=1)
print(f"single_feature_cv identical to low_novelty on every question: {dup} -> dropped from the family (family size {len(FAIR23)}, was 9)")
display(H2.round(4)); H2.to_csv(RES_DIR / "table_h2_holm_corrected.csv", index=False)

# budget sweep vs low_novelty, Holm across the 4 budgets
sw = []
for fr in sorted(R23.budget.unique()):
    ww = wide23("B", fr); sw.append(dict(budget=fr, **paired23(ww, PRIM, "low_novelty")))
SW = pd.DataFrame(sw); SW["holm_p"] = holm23(SW.p.values); SW["verdict"] = SW.apply(verdict23, axis=1)
print("\nLR minus low_novelty across budgets (Holm across the 4 budgets):"); display(SW.round(4))
SW.to_csv(RES_DIR / "table_budget_sweep_holm.csv", index=False)

# ---- (c) was the ablated optimizer term ever active? ----
P = R23[(R23.setting == "B") & (R23.method == PRIM) & np.isclose(R23.budget, PF_)].copy()
P["lam"] = P.params.map(lambda s: json.loads(s)["lam"]); P["gamma"] = P.params.map(lambda s: json.loads(s)["gamma"]); P["nu"] = P.params.map(lambda s: json.loads(s)["nu"])
fp = P.groupby(["seed", "fold"])[["lam", "gamma", "nu"]].first()
act = {"- redundancy penalty": "lam", "- cost-awareness": "gamma", "- temporal engine": "nu"}
rows = []
for abl, term in act.items():
    a = R23[(R23.setting == "B") & (R23.method == abl) & np.isclose(R23.budget, PF_)]
    m = P.merge(a, on=["seed", "fold", "qid"], suffixes=("_p", "_a")); m["d"] = m.recall_p - m.recall_a
    byf = m.groupby(["seed", "fold"]).d.mean()
    rows.append(dict(ablation=abl, term=term, folds_term_active=int((fp[term] > 0).sum()), folds=len(fp),
                     folds_with_any_difference=int((byf.abs() > 1e-12).sum()), mean_diff=m.d.mean(),
                     VALID_TEST_OF_COMPONENT=bool((fp[term] > 0).sum() > 0 and term != "gamma")))
ACT = pd.DataFrame(rows)
print("\nOptimizer-component audit (a component can only be 'tested' by an ablation if the primary actually used it):"); display(ACT.round(4))
ACT.to_csv(RES_DIR / "table_component_activity.csv", index=False)
print("Reading: gamma (cost-awareness) was 0 in every fold and lambda (redundancy) was 0 in every fold, so removing them cannot change anything by itself.")
print("The earlier 'cost-awareness SUPPORTED' (+0.003) comes entirely from the fold(s) where the primary chose nu=0.2 and the ablation did not:",
      fp[fp.nu > 0].index.tolist(), "-> a tuning-selection difference, not evidence for cost-aware selection.")

# ---- (d) abstention questions: 'evidence' that does not contain the answer ----
is_abs = lambda idx: ~pd.Index(idx).astype(str).str.endswith("_abs")
n_abs = int((~is_abs(w.index)).sum()); print(f"\nAbstention (_abs) questions in the file: {n_abs} of {len(w)}")
w2 = wide23(mask=is_abs)
ab = [dict(comparison=f"{PRIM} - {o}", subset="excluding _abs", **paired23(w2, PRIM, o)) for o in ["low_novelty", "random", "recency"]]
AB = pd.DataFrame(ab); AB["holm_p"] = holm23(AB.p.values)
mean_rows = pd.DataFrame({m: [w[m].mean(), w2[m].mean()] for m in [PRIM, "low_novelty", "recency", "random"]}, index=["all 500", "excluding _abs"])
print("Mean recall @10%:"); display(mean_rows.round(4)); display(AB.round(4)); AB.to_csv(RES_DIR / "table_abstention_sensitivity.csv", index=False)

# ---- (e) session-reuse leak path: exposure-gradient test ----
# exposure of a test question = share of its NON-gold sessions that also appear in some training-fold question's history.
sess_by_q = {r["question_id"]: (set(map(str, r["haystack_session_ids"])), set(map(str, r.get("answer_session_ids") or []))) for r in raw23}
split = R23[(R23.setting == "B") & (R23.method == "low_novelty") & np.isclose(R23.budget, PF_)][["seed", "fold", "qid"]].drop_duplicates()
rows = []
for (seed, fold), g in split.groupby(["seed", "fold"]):
    test = set(g.qid); train_sess = set().union(*[sess_by_q[q][0] for q in sess_by_q if q not in test])
    for q in test:
        s, gold = sess_by_q[q]; non = s - gold
        rows.append(dict(seed=seed, fold=fold, qid=q, exposure=len(non & train_sess) / max(len(non), 1)))
EXP = pd.DataFrame(rows).groupby("qid").exposure.mean().to_frame()
for m in [PRIM, "low_novelty", "random"]: EXP[m] = w[m]
EXP["tertile"] = pd.qcut(EXP.exposure, 3, labels=["low exposure", "mid", "high exposure"])
tt = EXP.groupby("tertile", observed=True)[[PRIM, "low_novelty", "random"]].mean(); tt.insert(0, "mean_exposure", EXP.groupby("tertile", observed=True).exposure.mean())
tt["LR minus low_novelty"] = tt[PRIM] - tt["low_novelty"]
rho_lr = stats.spearmanr(EXP.exposure, EXP[PRIM] - EXP["low_novelty"])
print("\nSession-reuse exposure test. If training exposure to a test history's sessions leaked into the learned model, LR should gain over the\n"
      "training-free rule (low_novelty) as exposure rises. A gradient near zero means no evidence of a leak.")
display(tt.round(4)); print(f"Spearman(exposure, LR - low_novelty) = {rho_lr.statistic:+.3f} (p = {rho_lr.pvalue:.3f}).")
print("NOTE: all 500 histories form ONE connected component through shared sessions, so a GroupKFold split is impossible; this gradient test replaces it.")
tt.to_csv(RES_DIR / "table_exposure_gradient.csv")

# ---- H2 robustness: the SAME comparison (LR vs low_novelty) under every defensible analysis choice ----
sig = pd.read_csv(RES_DIR / "table_significance.csv"); pp = pd.read_csv(RES_DIR / "table_pool_paired.csv"); lc = pd.read_csv(RES_DIR / "locomo_paired.csv")
o = sig[(sig.comparison == f"{PRIM} - low_novelty") & np.isclose(sig.budget, PF_)].iloc[0]
h = H2.set_index("comparison").loc[f"{PRIM} - low_novelty"]; a0 = AB.set_index("comparison").loc[f"{PRIM} - low_novelty"]
pl = pp[(pp.pool.str.startswith("same-pipeline")) & pp.comparison.str.endswith("low_novelty")].iloc[0]
lm = lc[lc.comparison.str.contains("trained on LongMemEval - low_novelty")].iloc[0]
SPEC = pd.DataFrame([
 dict(analysis="original v2.2 (family of 9, duplicate rule counted twice)", diff=o.mean_diff, ci_low=o.ci_low, ci_high=o.ci_high, p_adj=o.holm_p),
 dict(analysis="duplicate removed (family of 8)", diff=h.mean_diff, ci_low=h.ci_low, ci_high=h.ci_high, p_adj=h.holm_p),
 dict(analysis="Holm across the 4 budgets", diff=SW.set_index("budget").loc[PF_, "mean_diff"], ci_low=SW.set_index("budget").loc[PF_, "ci_low"], ci_high=SW.set_index("budget").loc[PF_, "ci_high"], p_adj=SW.set_index("budget").loc[PF_, "holm_p"]),
 dict(analysis="excluding 30 abstention questions (raw p)", diff=a0.mean_diff, ci_low=a0.ci_low, ci_high=a0.ci_high, p_adj=a0.p),
 dict(analysis="same-pipeline distractor pool (masked utilities)", diff=pl.mean_diff, ci_low=pl.ci_low, ci_high=pl.ci_high, p_adj=np.nan),
 dict(analysis="LoCoMo external test (conversation-cluster CI)", diff=lm.mean_diff, ci_low=lm.cluster_ci_low, ci_high=lm.cluster_ci_high, p_adj=np.nan)])
SPEC["CI_excludes_0"] = (SPEC.ci_low > 0) | (SPEC.ci_high < 0); SPEC["reliable (CI excl. 0 and adj p<.05)"] = SPEC.CI_excludes_0 & (SPEC.p_adj.fillna(1) < 0.05)
print("\nH2 robustness: learned LR minus low_novelty, same comparison under six analysis choices"); display(SPEC.round(4)); SPEC.to_csv(RES_DIR / "table_h2_spec_curve.csv", index=False)
n_rel = int(SPEC["reliable (CI excl. 0 and adj p<.05)"].sum())

# ---- corrected findings text ----
r = H2.set_index("comparison").loc[f"{PRIM} - low_novelty"]
add("MemWeave 2.3 corrected inference (Setting B, 10% budget). Replaces the H2/H3 verdicts in findings.txt; every number comes from a CSV written above.")
add(f"H2 vs best simple rule (low_novelty): diff {r.mean_diff:+.3f} [{r.ci_low:+.3f}, {r.ci_high:+.3f}]; Holm p = {r.holm_p:.3f} in the de-duplicated family of 8.")
add(f"   FRAGILE: the advantage is reliable under {n_rel} of {len(SPEC)} analysis choices (see table_h2_spec_curve.csv). Report it as a small, non-robust gain of about 0.03 recall, not as superiority.")
add("   Budget sweep (Holm across 4 budgets): " + "; ".join(f"{b:.0%}: p={p:.3f}" for b, p in zip(SW.budget, SW.holm_p)))
add("   single_feature_cv is the same rule as low_novelty (identical on all questions) and is no longer counted separately.")
add("H3: redundancy penalty and cost-awareness were never active (lambda = gamma = 0 in all 15 folds). Neither component is supported; the earlier +0.003 'cost-awareness' effect comes from one fold's nu choice.")
add(f"Abstention: {n_abs} _abs questions carry evidence labels that do not contain the answer; recall excluding them: LR {w2[PRIM].mean():.3f}, low_novelty {w2['low_novelty'].mean():.3f}, random {w2['random'].mean():.3f}.")
add(f"Session reuse: exposure gradient Spearman {rho_lr.statistic:+.3f} (p = {rho_lr.pvalue:.3f}); grouped CV is impossible because all histories are connected.")
(RES_DIR / "findings_v2_3.txt").write_text("\n".join(OUT)); print("\n" + "\n".join(OUT))


### 24. What can the thin evidence support?
LoCoMo minimum detectable effect, QA-proxy power, and the question-level link between retaining evidence and answering correctly.

In [ ]:
# ===== Section 24. What can the thin evidence support? Power, precision and the retention-to-answer link =====
import json, numpy as np, pandas as pd
from pathlib import Path
from scipy import stats
RES_DIR = globals().get("RES_DIR", Path("memweave2_results/v2_2")); PRIM = "MemWeave-LR (primary)"
R24 = globals().get("R", None)
if R24 is None: R24 = pd.read_csv(RES_DIR / "all_results_raw.csv")
rg = np.random.default_rng(24)

# ---- (1) LoCoMo: minimum detectable effect with 10 conversations ----
L = pd.read_csv(RES_DIR / "locomo_by_conversation.csv", index_col=0)
LR_L = "MemWeave-LR trained on LongMemEval"; RAND_L = "random (100-run mean)"
rows = []
for other in ["low_novelty", "recency", RAND_L]:
    d = (L[LR_L] - L[other]).to_numpy(); n = len(d); sd = d.std(ddof=1)
    mde = (stats.t.ppf(.975, n - 1) + stats.t.ppf(.80, n - 1)) * sd / np.sqrt(n)        # two-sided alpha .05, 80% power, paired t approximation
    rows.append(dict(comparison=f"LR - {other}", mean_diff=d.mean(), sd_across_conversations=sd, min_detectable_diff_80pct_power=mde,
                     LongMemEval_effect_vs_random=0.315 if other == RAND_L else np.nan))
LP = pd.DataFrame(rows)
print("LoCoMo power (10 conversations): the smallest true difference the test would detect 80% of the time"); display(LP.round(4))
mde_rand = float(LP.iloc[2].min_detectable_diff_80pct_power)
print(f"Reading: LR beat random by +0.315 on LongMemEval. On LoCoMo the test could detect a gain as small as {mde_rand:.3f}, and the estimate is {LP.iloc[2].mean_diff:+.3f}.\n"
      "So 'no transfer' is a well-powered conclusion for the LongMemEval-sized effect, but the +/-0.03 'equivalence' verdicts sit at the edge of what 10 conversations can resolve.")
LP.to_csv(RES_DIR / "table_locomo_power.csv", index=False)

# ---- (2) QA proxy: how many questions carry any information? ----
Q = pd.read_csv(RES_DIR / "qa_proxy_results.csv"); W = Q.pivot(index="qid", columns="method", values="correct")
n = len(W); informative = int((W.max(axis=1) == 1).sum()); never = n - informative
print(f"QA proxy: {n} questions | answered correctly by at least one policy: {informative} | wrong under every policy: {never}")
print("Fraction of answers that were 'unknown':", Q.answer_given.str.lower().str.contains("unknown").groupby(Q.method).mean().round(2).to_dict())
rows = []
for o in [c for c in W.columns if c != PRIM]:
    b = int(((W[PRIM] == 1) & (W[o] == 0)).sum()); c = int(((W[PRIM] == 0) & (W[o] == 1)).sum()); disc = (b + c) / n
    p = float(stats.binomtest(b, b + c, .5).pvalue) if b + c else 1.0
    delta = 0.05
    need = int(np.ceil((1.96 * np.sqrt(disc) + 0.8416 * np.sqrt(max(disc - delta ** 2, 1e-9))) ** 2 / delta ** 2)) if disc > delta ** 2 else np.nan
    rows.append(dict(vs=o, LR_only_correct=b, other_only_correct=c, exact_mcnemar_p=p, questions_needed_for_5pt_gap=need))
QA = pd.DataFrame(rows); print("\nPaired exact McNemar tests on the QA proxy:"); display(QA.round(3)); QA.to_csv(RES_DIR / "table_qa_power.csv", index=False)
print("Reading: only the comparison with full-history BM25 is reliable. The proxy cannot rank the memory policies against each other; ~"
      f"{int(QA.questions_needed_for_5pt_gap.min())}-{int(QA.questions_needed_for_5pt_gap.max())} questions would be needed to resolve a 5-point gap with this answer model.")

# ---- (3) does retaining evidence turn into correct answers? (same policy, same question) ----
S = R24[(R24.setting == "B") & np.isclose(R24.budget, .10) & (R24.seed == R24.seed.min())]          # seed used to build RETAINED in Section 11
S = S[S.method.isin(["recency", "random", "low_novelty", PRIM])][["qid", "method", "recall"]]
J = Q.merge(S, on=["qid", "method"], how="inner")
J["any_gold_retained"] = J.recall > 0; J["all_gold_retained"] = J.recall >= 1.0
tab = J.groupby("any_gold_retained").correct.agg(["mean", "sum", "count"]).rename(columns={"mean": "accuracy", "sum": "n_correct", "count": "n"})
print("\nQA accuracy when the policy retained at least one evidence session vs none (same questions, four policies pooled):"); display(tab.round(3))
t = J.groupby("any_gold_retained").correct.sum().reindex([True, False]).fillna(0); n_t = J.groupby("any_gold_retained").correct.count().reindex([True, False])
if (n_t > 0).all():
    p_fisher = stats.fisher_exact([[t[True], n_t[True] - t[True]], [t[False], n_t[False] - t[False]]])[1]
    print(f"Fisher exact p = {p_fisher:.3g}. (Question-clustered; treat as descriptive.)")
by_m = J.groupby("method").agg(retention_recall=("recall", "mean"), qa_accuracy=("correct", "mean"), share_with_any_gold=("any_gold_retained", "mean"))
print("\nPer policy, on the 100-question QA sample (retention recall here is seed-100 only):"); display(by_m.round(3))
acc_hit, acc_miss = float(tab.loc[True, "accuracy"]), float(tab.loc[False, "accuracy"])
print(f"Reading: at the question level, retaining an evidence session is associated with {acc_hit:.0%} QA accuracy versus {acc_miss:.0%} without one ({acc_hit / max(acc_miss, 1e-9):.1f}x), so retention recall is not decoupled from answers.\n"
      f"But policy-level QA accuracy ({by_m.qa_accuracy.min():.2f} to {by_m.qa_accuracy.max():.2f}) is inside the noise of 100 questions (see McNemar above), so the proxy cannot rank policies.\n"
      "Caveat: a few 'correct' answers without retained evidence (containment can match short answers by chance) and question difficulty both confound this pooled comparison.")
by_m.to_csv(RES_DIR / "table_retention_vs_qa.csv")


### 25. Audit-first presentation
Headline figure (mechanism, lift over random, features) and a ledger of what each MemWeave component contributes. Needs Section 23 to have run (uses `ACT`).

In [ ]:
# ===== Section 25. Audit-first presentation: headline figure and component ledger (reads saved CSVs only) =====
import numpy as np, pandas as pd, matplotlib
from pathlib import Path
import matplotlib.pyplot as plt
RES_DIR = globals().get("RES_DIR", Path("memweave2_results/v2_2")); PRIM = "MemWeave-LR (primary)"
EX = RES_DIR / "locomo_exploratory"
lme = pd.read_csv(RES_DIR / "table_evidence_group_B.csv").set_index("evidence_group")
loc = pd.read_csv(EX / "recall_by_evidence_group.csv").set_index("evidence_group")
nn_l = pd.read_csv(RES_DIR / "table_nearest_neighbour.csv"); nn_c = pd.read_csv(EX / "nearest_neighbour_locomo.csv")
auc_l = pd.read_csv(RES_DIR / "table_feature_auc.csv").set_index("feature"); auc_c = pd.read_csv(EX / "feature_auc_by_group.csv").set_index("feature")
LR_C = "MemWeave-LR trained on LongMemEval"; RND_C = "random (100-run mean)"
G1, G2 = "1 evidence session", "2+ evidence sessions"

# lift over random by dataset x evidence group (the quantity that is comparable across datasets)
lift = pd.DataFrame({
    "low_novelty (no learning)": [lme.loc[G2, "low_novelty"] - lme.loc[G2, "random"], lme.loc[G1, "low_novelty"] - lme.loc[G1, "random"],
                                  loc.loc[G2, "low_novelty"] - loc.loc[G2, RND_C], loc.loc[G1, "low_novelty"] - loc.loc[G1, RND_C]],
    "MemWeave-LR (learned)":    [lme.loc[G2, PRIM] - lme.loc[G2, "random"], lme.loc[G1, PRIM] - lme.loc[G1, "random"],
                                  loc.loc[G2, LR_C] - loc.loc[G2, RND_C], loc.loc[G1, LR_C] - loc.loc[G1, RND_C]]},
    index=["LongMemEval\n2+ evidence", "LongMemEval\n1 evidence", "LoCoMo\n2+ evidence", "LoCoMo\n1 evidence"])
print("Recall lift over random at 10% budget (comparable across datasets):"); display(lift.round(3)); lift.to_csv(RES_DIR / "table_audit_lift.csv")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
# (a) mechanism: do evidence sessions find each other?
obs = [nn_l.observed.mean(), nn_c.observed.mean()]; cha = [nn_l.chance.mean(), nn_c.chance.mean()]; x = np.arange(2)
ax[0].bar(x - .2, obs, .4, label="observed"); ax[0].bar(x + .2, cha, .4, label="chance")
for i in range(2): ax[0].text(x[i] - .2, obs[i] + .02, f"{obs[i]:.2f}", ha="center"); ax[0].text(x[i] + .2, cha[i] + .02, f"{cha[i]:.2f}", ha="center")
ax[0].set_xticks(x); ax[0].set_xticklabels(["LongMemEval-S", "LoCoMo"]); ax[0].set_ylim(0, 1); ax[0].legend()
ax[0].set_title("(a) Is an evidence session's nearest neighbour\nanother evidence session? (2+ evidence questions)", fontsize=10)
# (b) the headline: a rule with no learning captures the whole effect, and only on LongMemEval
lift.plot(kind="bar", ax=ax[1], rot=0); ax[1].axhline(0, color="k", lw=.8); ax[1].set_ylabel("recall minus random")
ax[1].set_title("(b) Lift over random @10% budget:\nno-learning rule vs learned model", fontsize=10); ax[1].legend(fontsize=8)
# (c) the features behind it (2+ evidence questions)
feats = ["novelty", "centrality"]; xs = np.arange(len(feats))
ax[2].bar(xs - .2, [auc_l.loc[f, "auc_2plus_evidence"] for f in feats], .4, label="LongMemEval-S"); ax[2].bar(xs + .2, [auc_c.loc[f, G2] for f in feats], .4, label="LoCoMo")
ax[2].axhline(.5, color="k", lw=.8); ax[2].set_xticks(xs); ax[2].set_xticklabels(feats); ax[2].set_ylim(0, 1); ax[2].set_ylabel("within-history AUC"); ax[2].legend(fontsize=8)
ax[2].set_title("(c) Pairwise-similarity features separate evidence\nfrom distractors only on LongMemEval-S", fontsize=10)
plt.tight_layout(); plt.savefig(RES_DIR / "fig_audit_headline.png", dpi=160); plt.close()
print("Saved", RES_DIR / "fig_audit_headline.png")

# ---- component ledger: what does each MemWeave component actually contribute? ----
ab = pd.read_csv(RES_DIR / "table_ablation_paired.csv")
ab["component"] = ab.comparison.str.replace(f"{PRIM} - - ", "", regex=False).str.replace(f"{PRIM} - ", "", regex=False)
ab["contributes"] = np.where((ab.ci_low > 0) & (ab.holm_p < .05), "YES (CI > 0, Holm p < .05)", "no reliable contribution")
ACT_ = globals().get("ACT", None)
if ACT_ is not None:
    inactive = {"redundancy penalty": "never active (lambda=0 in all folds): ablation is vacuous", "cost-awareness": "never active (gamma=0 in all folds): ablation is vacuous"}
    ab["note"] = ab.component.map(lambda c: inactive.get(c, ""))
else: ab["note"] = ""
ledger = ab[["component", "mean_diff", "ci_low", "ci_high", "holm_p", "contributes", "note"]]
print("\nComponent ledger (primary minus ablated; positive = component helps):"); display(ledger.round(4)); ledger.to_csv(RES_DIR / "table_component_ledger.csv", index=False)
kept = ledger[ledger.contributes.str.startswith("YES")].component.tolist()
print("\nComponents with a reliable contribution:", kept if kept else "none")
print("Paper framing that these numbers support: lead with panels (a)-(c), then report that a one-line, training-free rule reaches most of the learned model's recall,\n"
      "and that the six-component architecture is a baseline, not the contribution.")


### 26. Novelty ledger
Reproducible artifact statistics from the dataset file and a related-work checklist. Nothing is marked verified until the full text is read.

In [ ]:
# ===== Section 26. Novelty ledger: reproducible artifact statistics + related-work checklist =====
import json, collections, numpy as np, pandas as pd
from pathlib import Path
DATA_PATH = globals().get("DATA_PATH", Path("data/longmemeval/longmemeval_s_cleaned.json")); RES_DIR = globals().get("RES_DIR", Path("memweave2_results/v2_2"))
raw26 = globals().get("raw", None)
if raw26 is None: raw26 = json.load(open(DATA_PATH, encoding="utf-8"))

# ---- (1) numbers a reader can reproduce from the public file in seconds ----
gold = [(r["question_id"], str(g)) for r in raw26 for g in (r.get("answer_session_ids") or [])]
ids = [str(i) for r in raw26 for i in r["haystack_session_ids"]]; gold_set = {(q, g) for q, g in gold}
nongold = [str(i) for r in raw26 for i in r["haystack_session_ids"] if (r["question_id"], str(i)) not in gold_set]
abs_q = [r for r in raw26 if r["question_id"].endswith("_abs")]
fact = pd.DataFrame([
    dict(statement="gold sessions whose ID starts with 'answer_'", value=f"{sum(g.startswith('answer_') for _, g in gold)} / {len(gold)}"),
    dict(statement="non-gold sessions whose ID starts with 'answer_'", value=f"{sum(i.startswith('answer_') for i in nongold)} / {len(nongold)}"),
    dict(statement="abstention (_abs) questions that still list evidence sessions", value=f"{sum(bool(r['answer_session_ids']) for r in abs_q)} / {len(abs_q)} ({sum(len(r['answer_session_ids']) for r in abs_q)} sessions)"),
    dict(statement="questions with 2+ evidence sessions", value=f"{sum(len(r['answer_session_ids']) >= 2 for r in raw26)} / {len(raw26)}")])
nnl = pd.read_csv(RES_DIR / "table_nearest_neighbour.csv")
fact.loc[len(fact)] = dict(statement="2+ evidence: nearest neighbour of an evidence session is another evidence session (observed vs chance)", value=f"{nnl.observed.mean():.3f} vs {nnl.chance.mean():.3f}")
print("Reproducible artifact statistics (cite these, with the dataset file's SHA-256 from run_manifest.json):"); display(fact); fact.to_csv(RES_DIR / "table_artifact_facts.csv", index=False)

# ---- (2) related-work ledger. Entries come from searches run on 2026-11-10; NOTHING below is verified until full_text_read = True ----
LEDGER = pd.DataFrame([
 dict(work="Auditing Long-Term Memory Evaluation: Repeated Judging, Reader Variation, and Negative Controls", where="arXiv 2609.38021",
      covers="judge noise, reader variation and answer-key defects on LongMemEval-S (answer level)", overlap="audit framing and LongMemEval-S", gap_vs_yours="answer-level; no retention-budget or ID/clustering artifact seen in the excerpt", full_text_read=False),
 dict(work="UTILMEM", where="arXiv 2608.30508", covers="new benchmark with strong, semantically adjacent distractors; compares to LongMemEval-S",
      overlap="motivates harder distractors, i.e. same concern as your same-pipeline pool", gap_vs_yours="builds a benchmark; excerpt does not report an ID leak or evidence clustering", full_text_read=False),
 dict(work="TIAP: scoring-target audit", where="arXiv 2605.24060", covers="re-scores saved retrievals under raw/source/canonical targets on LoCoMo and LongMemEval-S",
      overlap="audit of evaluation design on both of your datasets", gap_vs_yours="scoring targets, not retention or distractor structure", full_text_read=False),
 dict(work="ReMe_longmemeval_clean_s_v2 (reviewed labels)", where="huggingface.co/datasets/agentscope-ai/ReMe_longmemeval_clean_s_v2", covers="reviewed answers and evidence IDs; evidence references 948 -> 819 across 58 records",
      overlap="DIRECTLY affects your labels (your file has 948 gold sessions)", gap_vs_yours="label correction, not an artifact analysis", full_text_read=False),
 dict(work="LoCoMo audit (Penfield Labs blog)", where="penfieldlabs.substack.com", covers="answer-key errors in LoCoMo; argues LongMemEval-S fits in current context windows",
      overlap="critique of both benchmarks", gap_vs_yours="blog, not peer reviewed; answer key and context size, not retention shortcuts", full_text_read=False)])
display(LEDGER); LEDGER.to_csv(RES_DIR / "table_related_work_ledger.csv", index=False)

# ---- (3) searches still to run by hand before any 'first to show' wording ----
TODO = ["GitHub issues and discussions on xiaowu0162/LongMemEval for 'answer_' prefix, session id, leakage, shortcut",
        "Google Scholar / arXiv: 'LongMemEval' + (shortcut OR artifact OR leakage OR spurious OR 'distractor')",
        "arXiv: query-agnostic / budgeted memory retention for LLM agents (the 'what to keep' framing; check MemGPT/Mem0/A-MEM follow-ups)",
        "Read the full text of every ledger row above and set full_text_read = True only after that"]
print("\nStill to do by hand:"); [print(" -", t) for t in TODO]
print("\nSafe wording until those searches are done: 'In the cleaned LongMemEval-S file we examined, ...' (a statement about the file),\n"
      "not 'we are the first to show ...'. The ID-prefix and abstention facts are verifiable from the file itself, so they are safe to state as observations.")

# ---- (4) optional: compare your labels with the reviewed release if you download it (not run here: no network in this session) ----
REVIEWED = Path("data/longmemeval/ReMe_longmemeval_clean_s_v2.json")
if REVIEWED.exists():
    rv = {r["question_id"]: set(map(str, r.get("answer_session_ids") or [])) for r in json.load(open(REVIEWED, encoding="utf-8"))}
    mine = {r["question_id"]: set(map(str, r.get("answer_session_ids") or [])) for r in raw26}
    changed = [q for q in mine if q in rv and rv[q] != mine[q]]
    print(f"\nReviewed labels differ on {len(changed)} questions; evidence sessions {sum(map(len, mine.values()))} -> {sum(len(rv[q]) for q in mine if q in rv)}.")
    print("To test sensitivity, set raw to the reviewed file, rebuild the cache (gold labels are cached), and rerun Sections 11-17 and 23.")
else:
    print(f"\n(Optional, not run) {REVIEWED} not found. Download the reviewed file to quantify how many of your gold labels change.")
